### 加载数据

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")

In [ ]:
xt = QmtProvider(dividend_type='back', timeout=180)

In [ ]:
sh_funds_list = xt.get_stock_list_in_sector("沪市基金")
sz_funds_list = xt.get_stock_list_in_sector("深市基金")
hs_funds_list = xt.get_stock_list_in_sector("沪深基金")

In [ ]:
start_date = "2005-01-01"
end_date = "2026-08-16"

In [ ]:
sh_funds_pl = xt.fetch_batch_ohlcv(sh_funds_list,start_date, end_date)
sh_funds_pl.write_parquet(f"sh_funds_{start_date.replace('-','')},{end_date.replace('-','')}.parquet")

In [ ]:
sz_funds_pl = xt.fetch_batch_ohlcv(sz_funds_list,start_date, end_date)
sz_funds_pl.write_parquet(f"sz_funds_{start_date.replace('-','')},{end_date.replace('-','')}.parquet")

In [ ]:
hs_funds_pl = xt.fetch_batch_ohlcv(hs_funds_list,start_date, end_date)
hs_funds_pl.write_parquet(f"hs_funds_{start_date.replace('-','')},{end_date.replace('-','')}.parquet")

In [ ]:
sh_funds_prices = (
    sh_funds_pl.select(["timestamp", "symbol", "close"])
    .pivot(on="symbol", index="timestamp", values="close", aggregate_function="first")
    .sort("timestamp")
)
sh_funds_prices.write_parquet(f"sh_funds_prices.parquet")

sz_funds_prices = (
    sz_funds_pl.select(["timestamp", "symbol", "close"])
    .pivot(on="symbol", index="timestamp", values="close", aggregate_function="first")
    .sort("timestamp")
)
sz_funds_prices.write_parquet(f"sz_funds_prices.parquet")

In [ ]:
hs_funds_prices = (
    hs_funds_pl.select(["timestamp", "symbol", "close"])
    .pivot(on="symbol", index="timestamp", values="close", aggregate_function="first")
    .sort("timestamp")
)
hs_funds_prices.write_parquet(f"hs_funds_prices.parquet")

In [ ]:
import numpy as np
import pandas as pd

MIN_PRICE = 0.01  # 前复权下低于 1 分钱视为不可用，按你的资产类型调整

def cut_unusable_head(s, min_price=MIN_PRICE):
    ok = s[s > min_price]
    if ok.empty:
        return pd.Series(np.nan, index=s.index, name=s.name)  # 整列都不合格，交给后续筛选剔除
    first_idx = ok.index[0]
    return s.where(s.index >= first_idx)

In [ ]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)

In [ ]:
import numpy as np

inf_cols = X.columns[np.isinf(X).any(axis=0)]
print(inf_cols.tolist())

In [ ]:
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

In [ ]:
from Pre_selection import DropTailCorrelated
from skfolio.pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated

### EqualWeighted基准

In [ ]:
from skfolio.optimization import EqualWeighted
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
```
purged_size=0：训练结束与测试开始无缝衔接。
purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。例如，purged_size=1 意味着当前时期的决策从下一个时期才开始影响性能。
建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。
```
- 训练集扩展与尾部数据处理
```
expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。
```

In [ ]:
from skfolio.model_selection import WalkForward

In [ ]:
train_portfolios = MultiPeriodPortfolio()
test_portfolios = MultiPeriodPortfolio()
complete_records = []
nondomin_records = []
tailcorr_records = []
cv = WalkForward(test_size=252//4, train_size=int(252*3), purged_size=1, reduce_test=True, expand_train=False)
for i, (train_index, test_index) in enumerate(cv.split(X)):
    # 划分训练测试集
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    # 完整性筛选
    complete_selector = SelectComplete(drop_assets_with_internal_nan=False)
    X_train = complete_selector.fit_transform(X_train)
    complete_records.append(len(X_train.columns))
    if X_train.empty or X_train.shape[1] < 10:
        continue
    # 0方差筛选
    variance_selector = DropZeroVariance(threshold=1e-8)
    X_train = variance_selector.fit_transform(X_train)
    if X_train.empty or X_train.shape[1] < 10:
        continue
    # 非支配筛选
    nondomin_selector = SelectNonDominated(min_n_assets=10,
                        fitness_measures=[PerfMeasure.MEAN, RiskMeasure.VARIANCE, RatioMeasure.SHARPE_RATIO])
    X_train = nondomin_selector.fit_transform(X_train)
    nondomin_records.append(len(X_train.columns))
    if X_train.empty or X_train.shape[1] < 10:
        continue
    # 相关性筛选
    corrlate_selector = DropCorrelated(threshold=0.1, absolute=False)
    X_train = corrlate_selector.fit_transform(X_train)
    # 尾部相关性筛选
    #tailcorr_selector = DropTailCorrelated(threshold=0.2, quantile=0.10)
    #X_train = tailcorr_selector.fit_transform(X_train)
    tailcorr_records.append(len(X_train.columns))
    if X_train.empty:
        continue
    
    m = EqualWeighted(portfolio_params=dict(name="Fold %d"%i)).fit(X_train)
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test[X_train.columns]))

population_train = Population(train_portfolios)
population_test = Population(test_portfolios)

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population = population_train + population_test

In [ ]:
complete_records,nondomin_records,tailcorr_records

In [ ]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [ ]:
population_train.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [ ]:
population_test.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [ ]:
population_test.plot_cumulative_returns()

In [ ]:
test_portfolios.plot_cumulative_returns()

In [ ]:
test_portfolios.summary()

### WalkForward + SyntheticData

In [ ]:
import numpy as np
import pandas as pd
from skfolio import Population, RiskMeasure
from skfolio.distribution import VineCopula
from skfolio.model_selection import WalkForward
from skfolio.optimization import EqualWeighted
from skfolio.portfolio import MultiPeriodPortfolio
from skfolio.pre_selection import (
    SelectComplete,
    DropZeroVariance,
    SelectNonDominated,
    DropCorrelated,
)
from skfolio.measures import PerfMeasure, RatioMeasure

# ---------- 合成数据组配置（可自由增删，方便后续比较） ----------
# 每组 = 一个不同的 VineCopula 配置，长度都等于该期测试集
SYNTH_SPECS = {
    "1": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "2": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "3": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "4": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "5": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "6": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "7": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "8": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "9": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "10": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    "11": dict(log_transform=True,  random_state=np.random.randint(1024), conditioning=None),
    "12": dict(log_transform=False, random_state=np.random.randint(1024), conditioning=None),
    # 压力/条件场景：conditioning 传 {资产名: 值或(下界,上界)}
    # "S3": dict(log_transform=True, random_state=2,
    #            conditioning={"某资产": -0.05}),
}
SYNTH_KEYS = list(SYNTH_SPECS.keys())

# ---------- 容器 ----------
train_portfolios = []
test_portfolios  = MultiPeriodPortfolio()
# 每组合成数据一个 MultiPeriodPortfolio
synth_portfolios = {k: MultiPeriodPortfolio() for k in SYNTH_KEYS}
for k in SYNTH_KEYS:
    synth_portfolios[k].tag = 'SYNTH'

complete_records = []
nondomin_records = []
tailcorr_records = []

cv = WalkForward(test_size=252//4, train_size=int(252 * 3),
                 purged_size=0, reduce_test=True, expand_train=True)

for i, (train_index, test_index) in enumerate(cv.split(X)):
    # 划分训练测试集
    X_train = X.iloc[train_index]
    X_test  = X.iloc[test_index]

    # 完整性筛选
    complete_selector = SelectComplete(drop_assets_with_internal_nan=False)
    X_train = complete_selector.fit_transform(X_train)
    complete_records.append(len(X_train.columns))
    if X_train.empty or X_train.shape[1] < 10:
        continue

    # 0方差筛选
    variance_selector = DropZeroVariance(threshold=1e-8)
    X_train = variance_selector.fit_transform(X_train)
    if X_train.empty or X_train.shape[1] < 10:
        continue

    # 非支配筛选
    nondomin_selector = SelectNonDominated(
        min_n_assets=10,
        fitness_measures=[PerfMeasure.MEAN, RiskMeasure.VARIANCE, RatioMeasure.SHARPE_RATIO],
    )
    X_train = nondomin_selector.fit_transform(X_train)
    nondomin_records.append(len(X_train.columns))
    if X_train.empty or X_train.shape[1] < 10:
        continue

    # 相关性筛选
    corrlate_selector = DropCorrelated(threshold=0.2, absolute=False)
    X_train = corrlate_selector.fit_transform(X_train)

    # 尾部相关性筛选
    # tailcorr_selector = DropTailCorrelated(threshold=0.2, quantile=0.10)
    # X_train = tailcorr_selector.fit_transform(X_train)
    tailcorr_records.append(len(X_train.columns))
    if X_train.empty:
        continue

    # 训练组合
    m = EqualWeighted(portfolio_params=dict(name="Fold %d" % i)).fit(X_train)

    # ① 真实测试集
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test[X_train.columns]))
    #print(X_train.min().min(), X_train.max().max())

    # ② 合成数据测试：每组一个，长度 = 测试集长度
    n_test = len(test_index)
    for g in SYNTH_KEYS:
        spec = SYNTH_SPECS[g]
        # 每期都用当期 X_train 重新拟合 VineCopula，保证资产列对齐
        vine = VineCopula(
            log_transform=spec["log_transform"],
            n_jobs=-1,
            random_state=spec["random_state"],
        )
        vine.fit(X_train)
        # 关键：vine.sample 返回 (n_test, n_assets) 的 ndarray
        synth_arr = vine.sample(n_samples=n_test, conditioning=spec.get("conditioning"))
        synth_arr = np.clip(synth_arr, -0.2, 0.2)
        #print(synth_arr.min().min(), synth_arr.max().max())
        # 包装成 DataFrame：索引 = X_test.index，列名 = X_train.columns
        synth_test = pd.DataFrame(
            synth_arr, index=X_test.index, columns=X_train.columns
        )
        # 用同一期权重在合成场景上计算组合
        synth_ptf = m.predict(synth_test)
        synth_ptf.name = f"Fold {i} {g}"
        synth_ptf.tag  = g                 # 用 tag 区分合成组
        synth_portfolios[g].append(synth_ptf)

# ---------- 汇总 ----------
population_train = Population(train_portfolios)
population_train.set_portfolio_params(tag="Train")

population_test = Population([test_portfolios])
population_test.set_portfolio_params(tag="Test")

# 把每组合成数据也加进同一个 Population，便于统一对比
#population = population_test
#for g in SYNTH_KEYS:
#    population = population + Population([synth_portfolios[g]])
#    population.filter(tags=g).set_portfolio_params(tag=g)

# 统一年化口径（日线=252；周线请改为 52 等）
#population.set_portfolio_params(annualized_factor=252)

# ---------- 对比 ----------
#population.filter(tags=["Test"] + SYNTH_KEYS).plot_cumulative_returns()

In [ ]:
Population([synth_portfolios[str(i)] for i in range(1, 13)]+[test_portfolios]).plot_cumulative_returns()

In [ ]:
population_test.plot_cumulative_returns()

### WalkForward+CombinatorialPurgedCV
组合净化交叉验证生成多个测试路径，以进行更稳健的时间序列模型评估和分布分析。

In [ ]:
from skfolio.model_selection import CombinatorialPurgedCV

In [ ]:
def nested_walkforward_cpcv(
    X: pd.DataFrame,
    outer_cv: WalkForward | None = None,
    inner_cv: CombinatorialPurgedCV | None = None,
    min_n_assets: int = 10,
    verbose: bool = True,
):
    """WalkForward(外) × CombinatorialPurgedCV(内) 嵌套交叉验证。

    外循环: WalkForward 滚动窗口, 每个 fold 把 train+test 拼成时间连续块;
    内循环: 在该连续块上做 CPCV, 每个组合都走预筛选→拟合→多测试块预测。
    """
    if outer_cv is None:
        outer_cv = WalkForward(
            test_size=252,
            train_size=int(252 * 3),
            purged_size=0,
            reduce_test=False,
            expand_train=False,
        )
    if inner_cv is None:
        inner_cv = CombinatorialPurgedCV(
            n_folds=8, n_test_folds=2, purged_size=0, embargo_size=0
        )
    result = {}
    path_result = {}

    train_portfolios = MultiPeriodPortfolio()   # 内层训练集组合
    test_path_ids = inner_cv.get_path_ids()     # 每个分割中每个测试集的路径 ID
    # 预筛选记录（与参考代码对应，附加 fold/split 标记）
    complete_records, nondomin_records, tailcorr_records = [], [], []
    skip_records = []

    for i, (train_index, test_index) in enumerate(outer_cv.split(X)):
        # ---- 1) train + test 拼成时间连续窗口 ----
        window_index = np.concatenate([train_index, test_index])
        # 若希望窗口严格连续（把外层 purged 掉的那 1 行也纳入）:
        # window_index = np.arange(train_index[0], test_index[-1] + 1)
        X_window = X.iloc[window_index]
        
        inner_result = dict()
        # test path 收集 (时间位置, portfolio)，稍后按时间排序重组
        path_holder = {i:[] for i in range(inner_cv.n_test_paths)}

        for j, (inner_train_idx, inner_test_idx_list) in enumerate(
            inner_cv.split(X_window)
        ):  
            # ---- 2) 相对窗口索引 -> 全局索引 ----
            abs_train_idx = window_index[inner_train_idx]
            X_train = X.iloc[abs_train_idx]

            # ---- 3) 预筛选（只在内层训练块上拟合） ----
            X_train = SelectComplete(
                drop_assets_with_internal_nan=False
            ).fit_transform(X_train)
            complete_records.append((i, j, len(X_train.columns)))
            if X_train.empty or X_train.shape[1] < min_n_assets:
                skip_records.append((i, j, "complete"))
                continue

            X_train = DropZeroVariance(threshold=1e-8).fit_transform(X_train)
            if X_train.empty or X_train.shape[1] < min_n_assets:
                skip_records.append((i, j, "zero_variance"))
                continue

            X_train = SelectNonDominated(
                min_n_assets=min_n_assets,
                fitness_measures=[
                    PerfMeasure.MEAN,
                    RiskMeasure.VARIANCE,
                ],
            ).fit_transform(X_train)
            nondomin_records.append((i, j, len(X_train.columns)))
            if X_train.empty or X_train.shape[1] < min_n_assets:
                skip_records.append((i, j, "nondominated"))
                continue

            X_train = DropCorrelated(threshold=0.1, absolute=False).fit_transform(
                X_train
            )
            # 尾部相关性筛选可按需打开
            # X_train = DropTailCorrelated(threshold=0.2, quantile=0.10).fit_transform(X_train)
            tailcorr_records.append((i, j, len(X_train.columns)))
            if X_train.empty:
                skip_records.append((i, j, "correlated"))
                continue

            # ---- 4) 拟合 ----
            m = EqualWeighted(
                portfolio_params=dict(name=f"WF{i}-CPCV{j}")
            ).fit(X_train)
            train_portfolios.append(m.predict(X_train))


            # ---- 5) 内层测试：每个组合有 n_test_folds 个测试块 ----
            ptfs = []
            for k, inner_test_idx in enumerate(inner_test_idx_list):
                abs_test_idx = window_index[inner_test_idx]
                ptf = m.predict(X.iloc[abs_test_idx][X_train.columns])
                ptf.name = f"WF{i}-CPCV{j}-Test{k}"
                ptfs.append(ptf)
            # ---- 收集测试path
            for path_id,portfolio in zip(test_path_ids[j], ptfs):
                path_holder[path_id].append(portfolio)
            # innner fold result
            fold_result = dict()
            fold_result['train_portfolio'] = m.predict(X_train)
            fold_result['test_population'] = Population(ptfs)
            fold_result['test_mportfolio'] = MultiPeriodPortfolio(ptfs)
            inner_result[j] = fold_result
        result[i] = inner_result
        path_result[i] = path_holder

        if verbose:
            print(
                f"Outer fold {i}: window={len(X_window)} obs | "
                f"inner splits={inner_cv.get_n_splits(X_window)} | "
                f"complete paths={len(path_holder)}"
            )

    return result, path_result

In [ ]:
res, paths = nested_walkforward_cpcv(X)

In [ ]:
res[0][0]

In [ ]:
Population([res[6][i]['train_portfolio'] for i in range(28)]).plot_cumulative_returns()

Merge all test paths

In [ ]:
test_fold_0 = Population([MultiPeriodPortfolio(paths[6][i]) for i in range(7)])
test_fold_0.plot_cumulative_returns()

All Test Path Combinations

In [ ]:
all_test_parts = [[MultiPeriodPortfolio(paths[i][j][-2:]) for j in range(7)] for i in range(7)]

In [ ]:
Population(all_test_parts[0]).plot_cumulative_returns()

In [ ]:
import numpy as np

def discrete_lhs_safe(nested_lists, n_samples, seed=None):
    """
    离散LHS采样 - 不要求元素可比较
    
    Parameters:
        nested_lists: List[List[Any]], 8个子列表，每个7个元素
        n_samples: int, 采样数量
        seed: int, 可选随机种子
    Returns:
        List[Tuple[Any, ...]], 每个元组是一个8元素组合
    """
    rng = np.random.default_rng(seed)
    n_factors = len(nested_lists)      # 8
    n_levels = len(nested_lists[0])    # 7
    
    # === 所有操作仅在整数索引上进行 ===
    lhs_indices = np.empty((n_samples, n_factors), dtype=np.intp)
    
    for j in range(n_factors):
        # Step 1: 生成 0..n_samples-1 的随机排列（纯整数操作）
        perm = rng.permutation(n_samples)
        
        # Step 2: 将排列映射到层级索引 0..6（纯整数除法）
        # 这等价于把 n_samples 个点均匀分配到 7 个桶中
        level_indices = (perm * n_levels) // n_samples
        
        lhs_indices[:, j] = level_indices
    
    # === 仅在最后一步通过索引访问实际对象 ===
    samples = []
    for i in range(n_samples):
        combo = tuple(
            nested_lists[j][lhs_indices[i, j]]   # 仅 __getitem__，无比较
            for j in range(n_factors)
        )
        samples.append(combo)
    
    return samples

In [ ]:
samples = discrete_lhs_safe(all_test_parts, n_samples=1000)

In [ ]:
Population([MultiPeriodPortfolio(sample) for sample in samples]).plot_cumulative_returns()

In [ ]:
import numpy as np
from scipy.stats.qmc import LatinHypercube

def discrete_lhs_safe(nested_lists, n_samples, seed=None):
    """
    离散LHS采样 - 基于scipy实现，不要求元素可比较
    
    Parameters:
        nested_lists: List[List[Any]], 各子列表长度可不等
        n_samples: int, 采样数量
        seed: int | None, 随机种子
    Returns:
        List[Tuple[Any, ...]]
    """
    n_factors = len(nested_lists)
    n_levels = np.array([len(lst) for lst in nested_lists])  # 支持各维度层级数不同

    # ① 连续空间 LHS（scipy 内部用优化算法，比手写 permutation 更均匀）
    sampler = LatinHypercube(d=n_factors, seed=seed)
    unit_samples = sampler.random(n=n_samples)  # (n_samples, n_factors) ∈ [0,1)

    # ② 向量化映射到离散层级索引（一行替代循环）
    level_indices = np.clip(
        (unit_samples * n_levels[np.newaxis, :]).astype(np.intp),
        0,
        n_levels[np.newaxis, :] - 1
    )  # (n_samples, n_factors)

    # ③ 构建 object 查找表 + 高级索引一次性取值
    # ③ 构建 object 查找表（逐元素赋值，避免触发 __array__ 协议）
    max_levels = int(n_levels.max())
    lookup = np.empty((n_factors, max_levels), dtype=object)
    for j, lst in enumerate(nested_lists):
        for k, obj in enumerate(lst):
            lookup[j, k] = obj          # ← 单个 object 赋值，安全

    col_idx = np.arange(n_factors)[np.newaxis, :]
    samples_array = lookup[col_idx, level_indices]

    return [tuple(row) for row in samples_array]

In [ ]:
all_test_parts[0]

In [ ]:
samples = discrete_lhs_safe(all_test_parts, 200)

In [ ]:
Population([MultiPeriodPortfolio(sample) for sample in samples]).plot_cumulative_returns()